# 📦 Olist E-Commerce Logistics Intelligence System
### **End-to-End Decision Intelligence & Predictive Analytics Project**

## 📖 Project Overview
This project transforms raw marketplace data from **Olist** into a sophisticated diagnostic tool for logistics optimization. Moving beyond descriptive reporting, this system implements **Proprietary Risk Scoring**, **AI-Powered Anomaly Detection**, and **Prescriptive Strategic Interventions** to identify regional bottlenecks and audit seller reliability.

## 🛠️ Tech Stack
* **Python (Pandas/NumPy/Scikit-Learn):** Used for the ETL pipeline, Weighted Risk Algorithms, and Statistical Modeling.
* **SQL (SQLite):** Leveraged for relational data warehousing and business logic persistence.
* **Power BI:** Targeted for executive dashboarding featuring Geospatial Heatmaps and AI-Augmented Forecasting.

## 🎯 Advanced Project Objectives
1. **Comprehensive Data Engineering:** Integrated **8 relational datasets** (Orders, Items, Products, Sellers, Customers, Reviews, Payments, and Translations) into a unified "Single Source of Truth".
2. **Proprietary Seller Risk Scoring:** Developed a **0–100 weighted algorithm** (Timeliness 40%, Quality 40%, Volume 20%) to segment vendors into Elite vs. Critical categories.
3. **Root Cause Analysis (RCA):** Engineered `shipping_route` and `delay_gap` features to pinpoint geographic and category-level delivery failures.
4. **Operational Intelligence:** Automated a **Strategic Intervention List** to flag specific high-variance routes (e.g., SP to BA) for immediate carrier audits.
5. **Predictive Readiness:** Engineered time-series datasets for **Weekly Performance Forecasting** and **KPI Anomaly Detection** using 7-day rolling Z-score logic.
---
## 🚀 Phase 1: Data Ingestion & Cleaning
*Starting by importing raw datasets and handling missing values...*

In [1]:
import pandas as pd

In [2]:
# Exact column names from the Olist dataset
date_cols = [
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Load the data
orders = pd.read_csv('../data/olist_orders_dataset.csv', parse_dates=date_cols)
items = pd.read_csv('../data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'], errors='coerce')
# Verify it worked
print("--- Check Column Types ---")
print(orders.dtypes[date_cols])

print("\n--- Check for Missing Values ---")
print(orders.isnull().sum())

--- Check Column Types ---
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

--- Check for Missing Values ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [3]:
# Create a 'clean' version of orders by removing rows where delivery date is missing
# .copy() ensures we don't accidentally change the original 'orders' dataframe
orders_clean = orders.dropna(subset=['order_delivered_customer_date']).copy()

#Summary of what happed?
print(f"Total Original Orders: {len(orders)}")
print(f"Orders after removing nulls: {len(orders_clean)}")
print(f"Removed Rows: {len(orders) - len(orders_clean)}")

Total Original Orders: 99441
Orders after removing nulls: 96476
Removed Rows: 2965


**Rows with missing delivery dates were removed because delivery delay cannot be calculated without both order and delivery timestamps.

In [4]:
# Convert timestamps to DATE ONLY for correct late logic
orders_clean['delivered_date'] = orders_clean['order_delivered_customer_date'].dt.date
orders_clean['estimated_date'] = orders_clean['order_estimated_delivery_date'].dt.date

# Calculate wait time (still correct with .dt.days)
orders_clean['actual_wait_time'] = (
    orders_clean['order_delivered_customer_date'] - 
    orders_clean['order_purchase_timestamp']
).dt.days

# Correct delay calculation (DATE-LEVEL comparison)
orders_clean['delay_days'] = (
    pd.to_datetime(orders_clean['delivered_date']) -
    pd.to_datetime(orders_clean['estimated_date'])
).dt.days

# Correct late flag
orders_clean['is_late'] = (orders_clean['delay_days'] > 0).astype(int)

#Outcome
print("Late Rate inside orders_clean:", orders_clean['is_late'].mean())
print(orders_clean[['actual_wait_time', 'delay_days', 'is_late']].head())

Late Rate inside orders_clean: 0.06773705377503213
   actual_wait_time  delay_days  is_late
0                 8          -8        0
1                13          -6        0
2                 9         -18        0
3                13         -13        0
4                 2         -10        0


In [5]:
# 1. Load ALL remaining tables to ensure no NameErrors or KeyErrors
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
translation = pd.read_csv('../data/product_category_name_translation.csv')

# 2. Map Portuguese category names to English
products = products.merge(translation, on='product_category_name', how='left')

# 3. The Comprehensive Merge (This creates your "Single Source of Truth")
df_merged = orders_clean.merge(items, on='order_id', how='left')
df_merged = df_merged.merge(products[['product_id', 'product_category_name_english', 'product_weight_g']], on='product_id', how='left')
df_merged = df_merged.merge(sellers, on='seller_id', how='left')
df_merged = df_merged.merge(customers, on='customer_id', how='left') # This fixes the KeyError
df_final = df_merged.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
# --- FIX LATE LOGIC INSIDE df_final (VERY IMPORTANT) ---

df_final['delivered_date'] = pd.to_datetime(df_final['order_delivered_customer_date']).dt.date
df_final['estimated_date'] = pd.to_datetime(df_final['order_estimated_delivery_date']).dt.date

df_final['delay_days'] = (
    pd.to_datetime(df_final['delivered_date']) -
    pd.to_datetime(df_final['estimated_date'])
).dt.days

df_final['is_late'] = (df_final['delay_days'] > 0).astype(int)
df_final['delay_cost_estimate'] = df_final['delay_days'].clip(lower=0) * 5  # or your chosen value
print("Correct Late Rate inside df_final:", df_final['is_late'].mean())
print(f"Success! Master dataset has {df_final.shape[0]} rows and {df_final.shape[1]} columns.")
print("Key columns now available: 'customer_state', 'seller_state', 'review_score'")

Correct Late Rate inside df_final: 0.06580716173909905
Success! Master dataset has 110839 rows and 30 columns.
Key columns now available: 'customer_state', 'seller_state', 'review_score'


In [6]:
# This helps analyze where delivery delays occur in the logistics chain.
# seller_handling_time → Time taken by the seller to hand over the order to the carrier
# carrier_transit_time → Time taken by the carrier to deliver the order to the customer
df_final['seller_handling_time'] = (
    pd.to_datetime(df_final['order_delivered_carrier_date']) -
    pd.to_datetime(df_final['order_approved_at'])
    ).dt.days

df_final['carrier_transit_time'] = (
    pd.to_datetime(df_final['order_delivered_customer_date']) -
    pd.to_datetime(df_final['order_delivered_carrier_date'])
    ).dt.days
df_final[['seller_handling_time','carrier_transit_time']].head()

,seller_handling_time,carrier_transit_time
0,2.0,6.0
1,0.0,12.0
2,0.0,9.0
3,3.0,9.0
4,0.0,1.0


In [7]:
# A. Correlation Feature: Is the product too heavy? (Heavier items often delay)
# Already added product_weight_g in the merge above.

# B. Logistics Route: Create a 'Source_to_Destination' string
df_final['shipping_route'] = df_final['seller_state'] + " to " + df_final['customer_state']

# C. Review Impact: Tag orders that were late AND got a bad review
df_final['late_and_bad_review'] = ((df_final['is_late'] == True) & (df_final['review_score'] <= 2)).astype(int)

In [8]:
import numpy as np

# 1. Group data by seller
seller_performance = df_final.groupby('seller_id').agg({
    'order_id': 'count',
    'is_late': 'mean',
    'review_score': 'mean'
}).rename(columns={'order_id': 'total_orders', 'is_late': 'late_rate', 'review_score': 'avg_rating'})

# 2. Filter for sellers with at least 5 orders for better accuracy
seller_performance = seller_performance[seller_performance['total_orders'] >= 5].copy()

# 3. Apply the Weights (Business Logic)
# Timeliness (40%) + Quality (40%) + Volume Consistency (20%)
seller_performance['timeliness_score'] = (1 - seller_performance['late_rate']) * 40
seller_performance['quality_score'] = (seller_performance['avg_rating'] / 5) * 40
seller_performance['volume_score'] = (np.log1p(seller_performance['total_orders']) / np.log1p(seller_performance['total_orders'].max())) * 20

# 4. Calculate Final Score (0-100)
seller_performance['final_score'] = seller_performance['timeliness_score'] + seller_performance['quality_score'] + seller_performance['volume_score']

# 5. Segment Sellers for Actionable Insights
def get_segment(score):
    if score >= 85: return 'Elite'
    if score >= 70: return 'Reliable'
    if score >= 50: return 'At-Risk'
    return 'Critical'

seller_performance['seller_category'] = seller_performance['final_score'].apply(get_segment)
print(seller_performance['seller_category'].value_counts())

seller_category
Reliable    1399
Elite        283
At-Risk      173
Critical      11
Name: count, dtype: int64


In [9]:
df_final['delay_cost_estimate'] = df_final['delay_days'].clip(lower=0) * 25

In [10]:
# 1. Create a 7-day rolling average for the Late Rate
df_final = df_final.sort_values('order_purchase_timestamp')
daily_late_rate = df_final.resample('D', on='order_purchase_timestamp')['is_late'].mean()

# 2. Calculate Z-Score (Statistical Anomaly)
rolling_mean = daily_late_rate.rolling(window=7).mean()
rolling_std = daily_late_rate.rolling(window=7).std().replace(0, np.nan)
z_score = (daily_late_rate - rolling_mean) / rolling_std

# 3. Flag anything more than 2 standard deviations away
anomalies = daily_late_rate[abs(z_score) > 2]
print(f"Detected {len(anomalies)} anomalies in the delivery pipeline.")

Detected 8 anomalies in the delivery pipeline.


In [11]:
# 1. Ensure timestamp is correct format
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])

# 2. Aggregate using the correct column name 'actual_wait_time'
weekly_performance = df_final.resample('W', on='order_purchase_timestamp').agg({
    'is_late': 'mean',
    'actual_wait_time': 'mean',  # This matches your Cell [4]
    'order_id': 'count'
}).rename(columns={'is_late': 'weekly_late_rate', 'actual_wait_time': 'avg_delivery_days'})

# 3. Export for Power BI
weekly_performance.to_csv('../data/weekly_forecast_data.csv')

print("✅ Success: Forecast-ready dataset exported to '../data/weekly_forecast_data.csv'")
display(weekly_performance.head())


✅ Success: Forecast-ready dataset exported to '../data/weekly_forecast_data.csv'


,weekly_late_rate,avg_delivery_days,order_id
order_purchase_timestamp,,,
2016-09-18,1.000000,54.000000,3
2016-09-25,NaN,NaN,0
2016-10-02,NaN,NaN,0
2016-10-09,0.007092,19.017730,282
2016-10-16,0.000000,19.170732,41


In [12]:
# Identify the Top 5 Bottleneck Routes for immediate action
top_bottlenecks = df_final[df_final['is_late'] == 1].groupby('shipping_route').agg({
    'order_id': 'count',
    'delay_days': 'mean'
}).sort_values('order_id', ascending=False).head(5)

print("--- STRATEGIC INTERVENTION LIST ---")
for route in top_bottlenecks.index:
    print(f"Action: Audit logistics carriers on route: {route}")

--- STRATEGIC INTERVENTION LIST ---
Action: Audit logistics carriers on route: SP to SP
Action: Audit logistics carriers on route: SP to RJ
Action: Audit logistics carriers on route: SP to MG
Action: Audit logistics carriers on route: SP to BA
Action: Audit logistics carriers on route: SP to RS


In [13]:
# Data Export & System Integration
import sqlite3

# 1. Create a database file for your SQL work
conn = sqlite3.connect('../data/olist_analysis.db')
df_final.to_sql('delivery_performance', conn, if_exists='replace', index=False)
conn.close()

# 2. Save a clean CSV for Power BI
df_final.to_csv('../data/olist_for_powerbi.csv', index=False)

print("Files created! Act 1 (Python) is finished.")

Files created! Act 1 (Python) is finished.


In [14]:
# --- EXPORT FOR POWER BI & SYSTEM INTEGRATION ---
import os

# 1. Create the 'data' folder if it doesn't exist
if not os.path.exists('../data'):
    os.makedirs('../data')

# 2. Add the missing 'volume_bucket' feature before export
# This categorizes orders by item quantity - required for Stage 1.3 of the roadmap
df_final['volume_bucket'] = pd.qcut(df_final['order_item_id'].rank(method='first'), 4, labels=['low','mid','high','very_high'])

# 3. Export the three essential files for Power BI
# The Master Table (Logistics & RCA)
df_final.to_csv('../data/olist_for_powerbi.csv', index=False)

# The Advanced Scoring Table (Seller Risk Audit)
seller_performance.to_csv('../data/seller_risk_scores.csv', index=True)

# The weekly trend table (Forecasting & Anomaly Detection)
weekly_performance.to_csv('../data/weekly_forecast_data.csv', index=True)

print("✅ Success! All 3 files for Power BI are now in your data folder.")

✅ Success! All 3 files for Power BI are now in your data folder.


In [15]:
import sqlite3
import pandas as pd

# 1. Connect and Save EVERYTHING to the Database
conn = sqlite3.connect('../data/olist_analysis.db')

# Save the main performance table
df_final.to_sql('delivery_performance', conn, if_exists='replace', index=False)

# Save the seller risk table (Critical for SQL Portfolio)
seller_performance.to_sql('seller_risk_audit', conn, if_exists='replace', index=True)

# 2. Check if all tables exist
table_check = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("--- Tables in Database ---")
print(table_check)

# 3. Validation Query: High-Risk Seller Audit
# This demonstrates to recruiters that your database solves real problems
test_query = """
SELECT seller_id, final_score, seller_category 
FROM seller_risk_audit 
WHERE seller_category = 'Critical' 
LIMIT 5;
"""
df_test = pd.read_sql(test_query, conn)

print("\n--- High-Risk Seller Audit Preview ---")
display(df_test)

conn.close()

--- Tables in Database ---
                   name
0  delivery_performance
1     seller_risk_audit

--- High-Risk Seller Audit Preview ---


,seller_id,final_score,seller_category
0,08cdbae123ff67ca4e36d9d641ce0119,36.327629,Critical
1,0d83f8e03188682112cc0d93523705cc,47.780344,Critical
2,30c7f28fd3a5897b2c82d152bb760c17,49.652532,Critical
3,427165bf50f8ca07efc7bdc2bfcf1688,46.313667,Critical
4,633ecdf879b94b5337cca303328e4a25,32.852532,Critical


# 🏁 Project Phase 1: Complete 
### **Data Engineering & Decision Intelligence Engine Ready**

The ETL (Extract, Transform, Load) pipeline is now fully operational. We have successfully converted raw, fragmented data into a unified analytical asset.

## 🧠 Strategic Deliverables
* **Master Data Integration:** Successfully merged **8 relational datasets** into a single "Source of Truth" (`olist_for_powerbi.csv`).
* **Proprietary Seller Audit:** Designed a **0-100 Weighted Risk Algorithm** that identifies 'Critical' vs 'Elite' vendors based on multi-dimensional performance.
* **Operational Alert System:** Implemented **Z-Score Anomaly Detection** to catch logistics failures as they happen.
* **Prescriptive Analytics:** Generated a **Strategic Intervention List** identifying the Top 5 bottleneck routes (e.g., SP to BA) for immediate carrier auditing.

## 📂 Final Data Persistence
The following assets have been generated in the `../data/` directory:
1. `olist_analysis.db`: Full relational SQLite database for SQL-based business queries.
2. `olist_for_powerbi.csv`: Master fact table for geospatial and root cause analysis.
3. `seller_risk_scores.csv`: Aggregated seller rankings for the vendor audit dashboard.
4. `weekly_forecast_data.csv`: Time-series dataset prepared for AI-augmented forecasting.

---
## 🚀 Next Step: Executive Dashboarding in Power BI
Moving to the visualization phase to build the **Logistics Command Center**, where these Python-engineered insights will be converted into interactive, real-time business tools.